# Band Openings Analysis

Visualize when bands open with hour×month heatmaps.

**What you'll learn:**
- Create heatmaps showing propagation by hour and month
- Compare band behavior across the solar cycle
- Identify optimal operating windows for specific paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import load_dataset, plot_band_heatmap

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

In [ ]:
# Load data
df = load_dataset("wspr")  # Or "contest", "rbn"
print(f"Loaded {len(df):,} signatures")

## Band ID Reference

| ID | Band | Frequency |
|----|------|----------|
| 102 | 160m | 1.8 MHz |
| 103 | 80m | 3.5 MHz |
| 105 | 40m | 7 MHz |
| 107 | 20m | 14 MHz |
| 109 | 15m | 21 MHz |
| 111 | 10m | 28 MHz |

In [ ]:
# Single band heatmap
plot_band_heatmap(df, band=107, title="20m Band Openings")
plt.show()

In [ ]:
# Multi-band comparison
BANDS = [107, 109, 111]  # 20m, 15m, 10m
BAND_NAMES = {107: "20m", 109: "15m", 111: "10m"}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, band in zip(axes, BANDS):
    band_df = df[df["band"] == band]
    pivot = band_df.groupby(["hour", "month"])["median_snr"].mean().unstack(fill_value=np.nan)
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn", center=0, cbar_kws={"label": "SNR (dB)"})
    ax.set_title(f"{BAND_NAMES[band]} Band")
    ax.set_xlabel("Month")
    ax.set_ylabel("Hour (UTC)")

plt.tight_layout()
plt.show()

In [ ]:
# Filter to specific path for targeted analysis
# Example: US West Coast to Europe
path_df = df[
    (df["tx_grid_4"].str.startswith("CM") | df["tx_grid_4"].str.startswith("DN")) &
    (df["rx_grid_4"].str.startswith("JN") | df["rx_grid_4"].str.startswith("JO"))
]

print(f"US West → Europe paths: {len(path_df):,}")

if len(path_df) > 100:
    plot_band_heatmap(path_df, band=107, title="20m: US West Coast → Europe")
    plt.show()

## Key Insights

**High bands (10m, 15m):**
- Open during daylight hours when F-layer ionization is high
- Seasonal: better in equinox months, solar maximum years
- Dead at night (MUF drops below band frequency)

**Mid bands (20m, 17m):**
- Workable 24 hours during solar maximum
- Best DX during greyline periods

**Low bands (40m, 80m, 160m):**
- Best at night when D-layer absorption fades
- Seasonal: winter nights have longer openings
- Greyline enhancement at sunrise/sunset